# HingRoBERTa-Mixed — One-Fold Macro-F1 Experiment

**Objective:** Evaluate whether a Hindi–English code-mixed pretrained encoder beats your existing XLM-RoBERTa Fold 0 Macro-F1 of **0.83370** *before* training more folds or creating any submission.

- **Model:** `l3cube-pune/hing-roberta-mixed`
- **Evaluation:** Same `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` and preprocessing as your earlier project; train only **fold 0**.
- **Rule:** Assert full model parameter count is below 300M **before training**. This tests a single standalone model; if your competition counts ensemble parameters, don't simply add it to your existing blend.
- **Cost control:** No test predictions, full-data training or `submission.csv` in this notebook. A best-fold checkpoint is saved only when it improves (optional).
- **Result:** Best validation Macro-F1 after each epoch and classification report at end. Scores are unknown until you run the experiment; 0.90 is not guaranteed.

Enable **GPU** and **Internet** in Kaggle settings; attach the competition data. If you already have the required packages, skip Step 1.

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/anveshan-data-sprint-2-nlp/sample_submission.csv
/kaggle/input/competitions/anveshan-data-sprint-2-nlp/train.csv
/kaggle/input/competitions/anveshan-data-sprint-2-nlp/test.csv


## 1. Install the tokenizer dependency

The checkpoint uses the XLM-RoBERTa tokenizer. Install `sentencepiece` if the Kaggle environment does not already have it.

In [5]:
%pip install -q "transformers==4.57.6" sentencepiece protobuf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries

We need PyTorch, Hugging Face Transformers, scikit-learn, and standard data-handling libraries.

In [3]:
import torch
import torch._utils

print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)
print("Internal utilities available:", hasattr(torch, "_utils"))

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Transformers imports successful!")

PyTorch version: 2.11.0+cu130
PyTorch location: /usr/local/lib/python3.13/dist-packages/torch/__init__.py
Internal utilities available: True
Transformers imports successful!


In [4]:
import os, re, glob, gc, random, time, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from tqdm.auto import tqdm


I0000 00:00:1795528824.780849     256 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
import sys
import torch
import transformers
import numpy as np

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)

Python: 3.13.16
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
NumPy: 2.1.3


In [6]:
import traceback
import sys
import transformers

print("Python version:", sys.version)
print("Transformers version:", transformers.__version__)

try:
    from transformers.models.auto.tokenization_auto import AutoTokenizer
    print("AutoTokenizer imported successfully!")

except Exception as e:
    print("\nIMPORT FAILED")

    root_error = e

    while root_error.__cause__ is not None:
        root_error = root_error.__cause__

    print("\nUnderlying error:")
    print(type(root_error).__name__, ":", root_error)

    print("\nDetailed traceback:")
    traceback.print_exception(
        type(root_error),
        root_error,
        root_error.__traceback__,
        limit=12
    )

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Transformers version: 4.57.6
AutoTokenizer imported successfully!


## 3. Set training configuration

This first experiment deliberately trains **one fold only**. Start with up to 8 epochs, monitoring Macro-F1 after every epoch. Settings are starting points—not guaranteed optimum values.

In [7]:
MODEL_NAME = "l3cube-pune/hing-roberta-mixed"
PARAM_LIMIT = 300_000_000
NUM_FOLDS = 5
FOLD_TO_RUN = 0
SEED = 42
EPOCHS = 8
PATIENCE = 2
BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
MAX_LENGTH = 64
LR_BODY = 3e-5
LR_HEAD = 1e-3
LABEL_SMOOTHING = 0.05
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
SAVE_BEST = True  # Saves only the best fold-0 checkpoint; set False if storage is limited
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."


## 4. Set seed and confirm GPU

Training this model on CPU will be very slow. This cell stops if Kaggle has not enabled a CUDA GPU.

In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
assert torch.cuda.is_available(), "Enable GPU in Kaggle Notebook settings before training."
device = torch.device("cuda")
USE_AMP = True
print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


## 5. Find and load the training dataset

Only `train.csv` is required for a validation experiment. If several candidate folders are attached, set `DATA_DIR` explicitly.

In [9]:
candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
candidates = [p for p in candidates if os.path.exists(os.path.join(os.path.dirname(p), "test.csv"))]
print("Candidate train.csv files:", candidates)
assert len(candidates) == 1, "Expected one matching folder. Set DATA_DIR manually if multiple are listed."
DATA_DIR = os.path.dirname(candidates[0])
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
assert {"query_text", "target"}.issubset(train_df.columns)
assert train_df["target"].notna().all(), "Missing labels found"
print("Shape:", train_df.shape)
print("Unique labels:", train_df.target.nunique())
train_df.head()


Candidate train.csv files: ['/kaggle/input/competitions/anveshan-data-sprint-2-nlp/train.csv']
Shape: (16936, 3)
Unique labels: 56


,example_id,query_text,target
0,100001,मैं चाहता हूं कि इस घटना को मेरे कैलेंडर से हट...,29
1,100002,आधा करने के लिए कम मात्रा,45
2,100003,kripaya rasoeeghar ki battiyaan band kar den,39
3,100004,vaikyum on karo,33
4,100005,ढाई घंटे के बाद घर को ऑटो लॉक करें,12


## 6. Normalize Hindi/Hinglish queries

This preprocessing matches your earlier XLM-R notebook so the Fold 0 comparison is more meaningful.

In [10]:
def norm(t):
    t = unicodedata.normalize("NFKC", str(t)).lower()
    t = t.replace("\u093c", "").replace("\u0901", "\u0902")
    t = re.sub(r"([a-z])\1{2,}", r"\1\1", t)
    t = re.sub(r"[^\w\s\u0900-\u097F]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

train_df["clean_text"] = train_df.query_text.fillna("").map(norm)
X = train_df.clean_text.tolist()
y = train_df.target.to_numpy(dtype=np.int64)
NUM_CLASSES = len(np.unique(y))
assert NUM_CLASSES == 56, f"Expected 56 classes, got {NUM_CLASSES}"
assert set(np.unique(y)) == set(range(NUM_CLASSES)), "Labels must be encoded 0..55"
print("Queries:", len(X), "| Classes:", NUM_CLASSES)
train_df[["query_text", "clean_text", "target"]].head()


Queries: 16936 | Classes: 56


,query_text,clean_text,target
0,मैं चाहता हूं कि इस घटना को मेरे कैलेंडर से हट...,मैं चाहता हूं कि इस घटना को मेरे कैलेंडर से हट...,29
1,आधा करने के लिए कम मात्रा,आधा करने के लिए कम मात्रा,45
2,kripaya rasoeeghar ki battiyaan band kar den,kripaya rasoeeghar ki battiyaan band kar den,39
3,vaikyum on karo,vaikyum on karo,33
4,ढाई घंटे के बाद घर को ऑटो लॉक करें,ढाई घंटे के बाद घर को ऑटो लॉक करें,12


## 7. Create exactly the same fold split

Use `StratifiedKFold` with seed 42 and evaluate **only fold 0**. No test-set data is needed here.

In [11]:
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
train_idx, val_idx = folds[FOLD_TO_RUN]
print("Training size:", len(train_idx), "Validation size:", len(val_idx))
print("Validation class count:", len(np.unique(y[val_idx])))


Training size: 13548 Validation size: 3388
Validation class count: 56


In [3]:
class_counts = np.bincount(
    y[train_idx],
    minlength=NUM_CLASSES
)

class_weights = 1.0 / np.sqrt(
    np.maximum(class_counts, 1)
)

class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)

NameError: name 'np' is not defined

## 8. Tokenize and create datasets

Dynamic padding saves memory by padding only to the longest sequence in each batch.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
encodings = tokenizer(X, truncation=True, max_length=MAX_LENGTH, padding=False)

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item

dataset = TextDataset(encodings, y)
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(Subset(dataset, train_idx.tolist()), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, num_workers=0, pin_memory=True)
val_loader = DataLoader(Subset(dataset, val_idx.tolist()), batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator, num_workers=0, pin_memory=True)
print("Training batches:", len(train_loader), "Validation batches:", len(val_loader))


tokenizer_config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Training batches: 1694 Validation batches: 212


## 9. Load the model and check the 300M parameter limit

This checkpoint is a **masked-language-model checkpoint**. The sequence-classification head is newly initialized and must be fine-tuned. For AMP stability, the model weights are explicitly kept in FP32. **Stop if the assertion fails.**

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)
model = model.float()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters (including classification head): {total_params / 1e6:.2f}M")
assert total_params <= PARAM_LIMIT, "STOP: over the competition's 300M parameter limit"
model = model.to(device)
print("Model device:", next(model.parameters()).device)
print("Model parameter dtype:", next(model.parameters()).dtype)
assert next(model.parameters()).dtype == torch.float32


config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-roberta-mixed and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total model parameters (including classification head): 278.09M
Model device: cuda:0
Model parameter dtype: torch.float32


## 10. Configure optimizer, scheduler, and mixed precision

Use a smaller learning rate for the pretrained encoder, a larger one for the new classifier head, and gradient accumulation for effective batch size 16.

In [14]:
encoder_params = [p for n, p in model.named_parameters() if "classifier" not in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": LR_BODY},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)
updates_per_epoch = (len(train_loader) + GRAD_ACCUMULATION - 1) // GRAD_ACCUMULATION
total_steps = EPOCHS * updates_per_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION)


Effective batch size: 16


## 11. Define one training epoch

This uses AMP with FP32 stored model parameters, gradient accumulation and clipping. The learning-rate scheduler advances only when the optimizer update was not skipped.

In [15]:
def train_one_epoch():
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
            full_loss = F.cross_entropy(logits.float(), labels, label_smoothing=LABEL_SMOOTHING)
        running_loss += full_loss.item()
        group_start = (step // GRAD_ACCUMULATION) * GRAD_ACCUMULATION
        group_size = min(GRAD_ACCUMULATION, len(train_loader) - group_start)
        scaler.scale(full_loss / group_size).backward()
        if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= scale_before:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    return running_loss / len(train_loader)


## 12. Define Macro-F1 validation

Evaluate only on held-out validation queries. Class scores are returned for detailed error analysis later.

In [16]:
@torch.no_grad()
def validate():
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(val_loader, desc="Validation", leave=False):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    preds = logits.argmax(axis=1)
    return f1_score(labels, preds, average="macro"), accuracy_score(labels, preds), logits, labels


## 13. Train just Fold 0 and print Macro-F1 every epoch

This is the only expensive step. It uses early stopping if Macro-F1 does not improve for two consecutive epochs (after at least four epochs). Optionally saves the best model for later use, so you do not lose a successful checkpoint.

In [17]:
best_f1, best_epoch, best_logits = -1.0, 0, None
history = []
no_improvement = 0
start = time.time()

for epoch in range(EPOCHS):
    train_loss = train_one_epoch()
    score, acc, logits, labels = validate()
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "macro_f1": score, "accuracy": acc})
    print(f"Epoch {epoch+1}/{EPOCHS}: loss={train_loss:.5f} | Macro-F1={score:.5f} | accuracy={acc:.5f}")
    if score > best_f1:
        best_f1, best_epoch, best_logits = score, epoch + 1, logits.copy()
        no_improvement = 0
        print("  New best Macro-F1")
        if SAVE_BEST:
            best_path = os.path.join(OUTPUT_DIR, "hingroberta_fold0_best")
            model.save_pretrained(best_path, safe_serialization=True)
            tokenizer.save_pretrained(best_path)
    else:
        no_improvement += 1
    if no_improvement >= PATIENCE and epoch + 1 >= 4:
        print("Early stopping: no improvement for", PATIENCE, "epochs")
        break

print(f"Fold 0 best Macro-F1: {best_f1:.5f} at epoch {best_epoch}")
print(f"Elapsed training time: {(time.time()-start)/60:.1f} minutes")


Training:   0%|          | 0/1694 [00:00<?, ?it/s]

Validation:   0%|          | 0/212 [00:00<?, ?it/s]

Epoch 1/8: loss=1.99407 | Macro-F1=0.72849 | accuracy=0.77686
  New best Macro-F1


Training:   0%|          | 0/1694 [00:00<?, ?it/s]

Validation:   0%|          | 0/212 [00:00<?, ?it/s]

Epoch 2/8: loss=1.04138 | Macro-F1=0.79211 | accuracy=0.83117
  New best Macro-F1


... (14 outputs left out)

Epoch 7/8: loss=0.44588 | Macro-F1=0.85458 | accuracy=0.87072
  New best Macro-F1


Training:   0%|          | 0/1694 [00:00<?, ?it/s]

Validation:   0%|          | 0/212 [00:00<?, ?it/s]

Epoch 8/8: loss=0.42676 | Macro-F1=0.85716 | accuracy=0.87367
  New best Macro-F1
Fold 0 best Macro-F1: 0.85716 at epoch 8
Elapsed training time: 23.2 minutes


## 14. Inspect the results before any submission work

Compare Fold 0 directly against your old XLM-R Fold 0 score of 0.83370. We are **not** yet measuring full five-fold OOF performance.

In [18]:
results_df = pd.DataFrame(history)
display(results_df)
print("New model Fold 0 best Macro-F1:", round(best_f1, 5))
print("Previous XLM-R Fold 0 Macro-F1: 0.83370")
print("Difference:", round(best_f1 - 0.83370, 5))
print("\nCLASSIFICATION REPORT — BEST EPOCH")
val_predictions = best_logits.argmax(axis=1)
print(classification_report(y[val_idx], val_predictions, digits=4, zero_division=0))
np.save(os.path.join(OUTPUT_DIR, "hingroberta_fold0_val_logits.npy"), best_logits)
pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "hingroberta_fold0_history.csv"), index=False)


,epoch,train_loss,macro_f1,accuracy
0,1,1.994074,0.728491,0.776860
1,2,1.041376,0.792113,0.831169
2,3,0.779774,0.828098,0.849764
3,4,0.635756,0.838687,0.857438
4,5,0.543227,0.845763,0.862751
5,6,0.481301,0.849597,0.867769
6,7,0.445885,0.854579,0.870720
7,8,0.426760,0.857159,0.873672


New model Fold 0 best Macro-F1: 0.85716
Previous XLM-R Fold 0 Macro-F1: 0.83370
Difference: 0.02346

CLASSIFICATION REPORT — BEST EPOCH
              precision    recall  f1-score   support

           0     0.8846    0.9200    0.9020       100
           1     0.8421    0.8421    0.8421        38
           2     0.9500    0.9744    0.9620        39
           3     0.9000    0.7297    0.8060        37
           4     1.0000    0.9318    0.9647        44
           5     0.7619    0.8421    0.8000        19
           6     0.9149    0.7818    0.8431        55
           7     0.9412    0.8889    0.9143        18
           8     0.8696    1.0000    0.9302        20
           9     0.7313    0.8167    0.7717        60
          10     0.9362    0.9362    0.9362        47
          11     1.0000    0.9394    0.9688        33
          12     0.6936    0.6630    0.6780       181
          13     0.9263    0.9832    0.9539       179
          14     1.0000    0.7692    0.8696        26

## Decision after Fold 0

- **Below ~0.83 Macro-F1:** No reason yet to run four more folds; inspect class-level errors or switch approach.
- **Around 0.83–0.85:** Similar to your existing XLM-R. A full five-fold evaluation may be informative but won't necessarily beat your current ensemble.
- **Meaningfully above 0.85:** Consider running all five folds and computing overall OOF Macro-F1 before deciding whether to make a submission.

**Important:** Fold 0 best-epoch selection is a screening metric, not an unbiased guarantee of a test-set or leaderboard score. The existing **0.84577** baseline is a five-fold *ensemble* OOF result and is not directly comparable with one validation fold. No submission file is produced here.

In [19]:
print("Best Macro-F1:", best_f1)
print("Best Epoch:", best_epoch)
print("Checkpoint:", best_path)
print("Checkpoint exists:", os.path.isdir(best_path))

Best Macro-F1: 0.8571585083416415
Best Epoch: 8
Checkpoint: /kaggle/working/hingroberta_fold0_best
Checkpoint exists: True


In [20]:
test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv")
)

test_df["clean_text"] = (
    test_df["query_text"].fillna("").map(norm)
)

test_encodings = tokenizer(
    test_df["clean_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

test_items = [
    {
        key: values[i]
        for key, values in test_encodings.items()
    }
    for i in range(len(test_df))
]

test_loader = DataLoader(
    test_items,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collator,
    pin_memory=True
)

print("Test samples:", len(test_df))

Test samples: 4912


In [21]:
del model, optimizer, scheduler, scaler
del encoder_params, head_params

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    best_path
).float().to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

assert total_params <= PARAM_LIMIT

print("Best model loaded successfully")
print(f"Parameters: {total_params / 1e6:.2f}M")

Best model loaded successfully
Parameters: 278.09M


In [22]:
@torch.no_grad()
def predict_test():

    model.eval()

    all_logits = []

    for batch in tqdm(
        test_loader,
        desc="Predicting"
    ):

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(**batch).logits

        all_logits.append(
            logits.float().cpu().numpy()
        )

    return np.concatenate(all_logits, axis=0)

In [23]:
test_logits = predict_test()

predictions = test_logits.argmax(axis=1)

submission = pd.DataFrame({
    "example_id": test_df["example_id"],
    "target": predictions
})

submission.to_csv(
    "/kaggle/working/hingroberta_fold0_submission.csv",
    index=False
)

print("Submission shape:", submission.shape)
print("Class 55 share:", (predictions == 55).mean())
print("Backup submission saved successfully!")

submission.head()

Predicting:   0%|          | 0/307 [00:00<?, ?it/s]

Submission shape: (4912, 2)
Class 55 share: 0.06820032573289903
Backup submission saved successfully!


,example_id,target
0,900001,32
1,900002,38
2,900003,43
3,900004,53
4,900005,48


In [24]:
FINAL_EPOCHS = best_epoch

print("Selected epochs:", FINAL_EPOCHS)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=True
)

print("Total training samples:", len(dataset))
print("Training batches:", len(train_loader))

Selected epochs: 8
Total training samples: 16936
Training batches: 2117


In [25]:
del model

gc.collect()
torch.cuda.empty_cache()

set_seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)

model = model.float().to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

assert total_params <= PARAM_LIMIT

print(f"Model parameters: {total_params / 1e6:.2f}M")
print("Final model initialized")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-roberta-mixed and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 278.09M
Final model initialized


In [26]:
encoder_params = [
    p for n, p in model.named_parameters()
    if "classifier" not in n
]

head_params = [
    p for n, p in model.named_parameters()
    if "classifier" in n
]

optimizer = torch.optim.AdamW([
    {
        "params": encoder_params,
        "lr": LR_BODY
    },
    {
        "params": head_params,
        "lr": LR_HEAD
    }
], weight_decay=WEIGHT_DECAY)

updates_per_epoch = (
    len(train_loader) + GRAD_ACCUMULATION - 1
) // GRAD_ACCUMULATION

total_steps = FINAL_EPOCHS * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(total_steps * WARMUP_RATIO),
    total_steps
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

In [27]:
start = time.time()

for epoch in range(FINAL_EPOCHS):

    train_loss = train_one_epoch()

    print(
        f"Epoch {epoch + 1}/{FINAL_EPOCHS} "
        f"| Loss: {train_loss:.5f} "
        f"| Elapsed: {(time.time()-start)/60:.2f} min"
    )

print("Final training completed!")

Training:   0%|          | 0/2117 [00:00<?, ?it/s]

Epoch 1/8 | Loss: 1.90344 | Elapsed: 3.54 min


Training:   0%|          | 0/2117 [00:00<?, ?it/s]

Epoch 2/8 | Loss: 1.01359 | Elapsed: 7.07 min


Training:   0%|          | 0/2117 [00:00<?, ?it/s]

Epoch 3/8 | Loss: 0.76953 | Elapsed: 10.60 min


... (6 outputs left out)

Training:   0%|          | 0/2117 [00:00<?, ?it/s]

Epoch 7/8 | Loss: 0.43744 | Elapsed: 24.76 min


Training:   0%|          | 0/2117 [00:00<?, ?it/s]

Epoch 8/8 | Loss: 0.42255 | Elapsed: 28.29 min
Final training completed!


In [28]:
final_logits = predict_test()

final_predictions = final_logits.argmax(axis=1)

submission = pd.DataFrame({
    "example_id": test_df["example_id"],
    "target": final_predictions
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Final submission shape:", submission.shape)
print("Class 55 share:", (final_predictions == 55).mean())
print("Final submission saved successfully!")

submission.head()

Predicting:   0%|          | 0/307 [00:00<?, ?it/s]

Final submission shape: (4912, 2)
Class 55 share: 0.06921824104234528
Final submission saved successfully!


,example_id,target
0,900001,21
1,900002,38
2,900003,43
3,900004,53
4,900005,48


In [29]:
model.save_pretrained(
    "/kaggle/working/hingroberta_final"
)

tokenizer.save_pretrained(
    "/kaggle/working/hingroberta_final"
)

print("Final model saved successfully!")

Final model saved successfully!


In [30]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

val_logits = best_logits.copy()
val_labels = y[val_idx]

assert len(val_logits) == len(val_labels)

original_f1 = f1_score(
    val_labels,
    val_logits.argmax(axis=1),
    average="macro"
)

print("Original Macro-F1:", original_f1)

Original Macro-F1: 0.8571585083416415


In [31]:
bias = np.zeros(val_logits.shape[1])
best_score = original_f1

# Class 55 threshold tuning
for threshold in np.arange(-2, 2.01, 0.05):

    candidate = bias.copy()
    candidate[55] = threshold

    score = f1_score(
        val_labels,
        (val_logits + candidate).argmax(axis=1),
        average="macro"
    )

    if score > best_score:
        best_score = score
        bias = candidate.copy()

print("After class 55 tuning:", best_score)

# Per-class bias tuning
for iteration in range(2):

    for c in range(val_logits.shape[1]):

        if c == 55:
            continue

        for adjustment in np.arange(-1.5, 1.51, 0.25):

            candidate = bias.copy()
            candidate[c] = adjustment

            score = f1_score(
                val_labels,
                (val_logits + candidate).argmax(axis=1),
                average="macro"
            )

            if score > best_score + 1e-5:
                best_score = score
                bias = candidate.copy()

    print(f"Iteration {iteration + 1}: {best_score:.5f}")

After class 55 tuning: 0.8571585083416415
Iteration 1: 0.87734
Iteration 2: 0.88046


In [32]:
final_bias = bias * 0.5

tuned_preds = (val_logits + final_bias).argmax(axis=1)

final_f1 = f1_score(
    val_labels,
    tuned_preds,
    average="macro"
)

final_accuracy = accuracy_score(
    val_labels,
    tuned_preds
)

print("=" * 50)
print(f"Original Macro-F1:   {original_f1:.5f}")
print(f"Fully tuned F1:      {best_score:.5f}")
print(f"Shrunk bias F1:      {final_f1:.5f}")
print(f"Final Accuracy:      {final_accuracy:.5f}")
print(f"F1 Improvement:     {final_f1-original_f1:.5f}")
print("=" * 50)

Original Macro-F1:   0.85716
Fully tuned F1:      0.88046
Shrunk bias F1:      0.86578
Final Accuracy:      0.87544
F1 Improvement:     0.00862


In [33]:
assert test_logits.shape == (
    len(test_df), len(final_bias)
)

tuned_test_predictions = (
    test_logits + final_bias
).argmax(axis=1)

tuned_submission = pd.DataFrame({
    "example_id": test_df["example_id"],
    "target": tuned_test_predictions
})

tuned_submission.to_csv(
    "/kaggle/working/hingroberta_tuned_submission.csv",
    index=False
)

print("Submission saved successfully!")

from IPython.display import FileLink

FileLink("hingroberta_tuned_submission.csv")

Submission saved successfully!


/kaggle/working/hingroberta_tuned_submission.csv

In [ ]:
from scipy.special import softmax

tta_text = (
    test_df["query_text"]
    .fillna("")
    .map(lambda t: unicodedata.normalize("NFKC", str(t)).lower().strip())
    .tolist()
)

tta_encodings = tokenizer(
    tta_text,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

tta_items = [
    {k: v[i] for k, v in tta_encodings.items()}
    for i in range(len(test_df))
]

original_test_loader = test_loader

test_loader = DataLoader(
    tta_items,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collator,
    pin_memory=True
)

try:
    tta_logits = predict_test()
    test_loader = original_test_loader

print("TTA completed!")